# Phase 4 - Notebook 07: Inference & Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/07_inference_evaluation.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Set up the environment for running MVSplat/pixelSplat inference
2. Run inference on test images and custom image pairs
3. Implement quantitative evaluation (PSNR, SSIM, LPIPS)
4. Benchmark inference speed and analyze failure cases
5. Visualize depth predictions and Gaussian distributions

**Estimated Time**: 60 minutes

**Prerequisites**: Notebook 06 (MVSplat Code Walkthrough)

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 1. Environment Setup

### 1.1 Installing MVSplat (Reference)

```bash
# Option 1: Full installation for inference
git clone https://github.com/donydchen/mvsplat.git
cd mvsplat
pip install -r requirements.txt
pip install gsplat==0.1.10

# Download pretrained model
mkdir -p pretrained
# From HuggingFace: https://huggingface.co/donydchen/mvsplat
```

### 1.2 Installing pixelSplat (Reference)

```bash
git clone https://github.com/dcharatan/pixelsplat.git
cd pixelsplat
pip install -e .
```

### 1.3 This Notebook

We use our simplified implementations from previous notebooks to demonstrate the evaluation pipeline. The same evaluation code works with official pretrained models.

## 2. Inference Pipeline

### 2.1 Complete Inference Steps

```
Input: 2 images + camera parameters
    ↓
Step 1: Preprocess
    - Resize to model input size (e.g., 256x256)
    - Normalize to [0, 1] or ImageNet stats
    - Prepare camera intrinsics/extrinsics tensors
    ↓
Step 2: Model forward pass
    - Extract features from both views
    - Build/process cost volume (MVSplat) or cross-attend (pixelSplat)
    - Predict per-pixel Gaussian parameters
    ↓
Step 3: Create Gaussians
    - Back-project depth to 3D positions
    - Apply activations (exp, sigmoid, normalize)
    - Merge Gaussians from all views
    ↓
Step 4: Render novel view
    - Specify target camera pose
    - Render via differentiable splatting
    ↓
Output: Rendered image from novel viewpoint
```

In [ ]:
from src.feedforward.gaussian_predictor import GaussianPredictionHeads
from src.feedforward.pixel_aligned import PixelAlignedGaussians, unproject_depth_to_3d
from src.feedforward.cost_volume import CostVolumeBuilder


class InferenceModel(nn.Module):
    """Simplified model for inference demonstration."""

    def __init__(self, feature_dim=32, num_depths=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, feature_dim, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(feature_dim, feature_dim, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.heads = GaussianPredictionHeads(
            in_channels=feature_dim, hidden_channels=16,
            depth_mode='regression', covariance_mode='3d',
        )

    def forward(self, image, K):
        B = image.shape[0]
        features = self.encoder(image)
        predictions = self.heads(features)
        depth = predictions['depth']
        positions = unproject_depth_to_3d(depth, K)
        _, _, H, W = depth.shape
        N = H * W
        return {
            'positions': positions.reshape(B, N, 3),
            'colors': image.flatten(2).permute(0, 2, 1),
            'opacities': predictions['opacities'].flatten(2).permute(0, 2, 1),
            'scales': predictions['scales'].flatten(2).permute(0, 2, 1),
            'rotations': predictions['rotations'].flatten(2).permute(0, 2, 1),
            'depth': depth,
        }


# Create model
model = InferenceModel()
model.eval()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Ready for inference.")

In [ ]:
# Generate synthetic test data

def create_synthetic_test_scene(H=64, W=64, num_views=4):
    """Create a synthetic test scene with known cameras."""
    fx, fy = 50.0, 50.0
    K = torch.tensor([[fx, 0, W/2], [0, fy, H/2], [0, 0, 1]], dtype=torch.float32)

    images = []
    poses = []

    for i in range(num_views):
        angle = i * 0.1
        tx = i * 0.3

        # Camera pose
        R = torch.tensor([
            [np.cos(angle), 0, np.sin(angle)],
            [0, 1, 0],
            [-np.sin(angle), 0, np.cos(angle)],
        ], dtype=torch.float32)
        pose = torch.eye(4)
        pose[:3, :3] = R
        pose[:3, 3] = torch.tensor([tx, 0, 0])
        poses.append(pose)

        # Synthetic image (textured pattern)
        torch.manual_seed(42)  # Same scene
        u = torch.linspace(-1, 1, W)
        v = torch.linspace(-1, 1, H)
        VV, UU = torch.meshgrid(v, u, indexing='ij')
        img = torch.stack([
            0.5 + 0.3 * torch.sin(UU * 5 + tx * 2),
            0.5 + 0.3 * torch.cos(VV * 5 + tx),
            0.5 + 0.2 * torch.sin((UU + VV) * 3),
        ], dim=0).clamp(0, 1)
        images.append(img)

    return {
        'images': torch.stack(images),  # [N, 3, H, W]
        'poses': torch.stack(poses),    # [N, 4, 4]
        'K': K,                          # [3, 3]
    }


test_scene = create_synthetic_test_scene(H=64, W=64, num_views=4)
print(f"Test scene: {test_scene['images'].shape[0]} views, "
      f"{test_scene['images'].shape[2]}x{test_scene['images'].shape[3]}")

# Visualize test views
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for i in range(4):
    axes[i].imshow(test_scene['images'][i].permute(1, 2, 0).numpy())
    label = 'Context' if i < 2 else 'Target'
    axes[i].set_title(f'View {i} ({label})', fontsize=10, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Test Scene Views (first 2 = context, last 2 = target)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Run inference

context_images = test_scene['images'][:2]  # 2 context views
target_images = test_scene['images'][2:]    # 2 target views
K = test_scene['K'].unsqueeze(0)  # [1, 3, 3]

with torch.no_grad():
    # Predict Gaussians from context view 0
    gaussians = model(context_images[0:1], K)

print("Inference results:")
for key, val in gaussians.items():
    print(f"  {key:12s}: {list(val.shape)}")

# Visualize predicted depth and Gaussians
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Input image
ax = axes[0]
ax.imshow(context_images[0].permute(1, 2, 0).numpy())
ax.set_title('Input (Context View 0)', fontsize=11, fontweight='bold')
ax.axis('off')

# Predicted depth
ax = axes[1]
depth = gaussians['depth'][0, 0].numpy()
im = ax.imshow(depth, cmap='plasma')
ax.set_title('Predicted Depth', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Gaussian positions (top view)
ax = axes[2]
pos = gaussians['positions'][0].numpy()  # [N, 3]
ax.scatter(pos[:, 0], pos[:, 2], c=pos[:, 2], cmap='plasma', s=2, alpha=0.5)
ax.set_xlabel('X'); ax.set_ylabel('Z (depth)')
ax.set_title(f'3D Gaussians (N={pos.shape[0]})', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Quantitative Evaluation

### 3.1 Evaluation Metrics Implementation

In [ ]:
class ImageMetrics:
    """
    Compute standard image quality metrics:
    PSNR, SSIM, and simplified LPIPS.
    """

    @staticmethod
    def psnr(predicted, target, max_val=1.0):
        """Peak Signal-to-Noise Ratio (dB)."""
        mse = (predicted - target).pow(2).mean()
        if mse < 1e-10:
            return float('inf')
        return (10 * torch.log10(max_val ** 2 / mse)).item()

    @staticmethod
    def ssim(predicted, target, window_size=11):
        """Structural Similarity Index."""
        C1 = 0.01 ** 2
        C2 = 0.03 ** 2
        C = predicted.shape[1]

        sigma = 1.5
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = g / g.sum()
        window = (g.unsqueeze(1) * g.unsqueeze(0)).unsqueeze(0).unsqueeze(0)
        window = window.expand(C, -1, -1, -1).to(predicted.device)
        pad = window_size // 2

        mu1 = F.conv2d(predicted, window, padding=pad, groups=C)
        mu2 = F.conv2d(target, window, padding=pad, groups=C)
        sigma1_sq = F.conv2d(predicted ** 2, window, padding=pad, groups=C) - mu1 ** 2
        sigma2_sq = F.conv2d(target ** 2, window, padding=pad, groups=C) - mu2 ** 2
        sigma12 = F.conv2d(predicted * target, window, padding=pad, groups=C) - mu1 * mu2

        ssim_map = ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)) / \
                   ((mu1 ** 2 + mu2 ** 2 + C1) * (sigma1_sq + sigma2_sq + C2))
        return ssim_map.mean().item()

    @staticmethod
    def lpips_approx(predicted, target):
        """Approximate LPIPS using gradient-based features."""
        # Compute image gradients as simple features
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                                dtype=torch.float32).reshape(1, 1, 3, 3)
        sobel_y = sobel_x.transpose(2, 3)
        sobel_x = sobel_x.expand(3, -1, -1, -1)
        sobel_y = sobel_y.expand(3, -1, -1, -1)

        grad_pred_x = F.conv2d(predicted, sobel_x, padding=1, groups=3)
        grad_pred_y = F.conv2d(predicted, sobel_y, padding=1, groups=3)
        grad_tgt_x = F.conv2d(target, sobel_x, padding=1, groups=3)
        grad_tgt_y = F.conv2d(target, sobel_y, padding=1, groups=3)

        diff_x = (grad_pred_x - grad_tgt_x).pow(2).mean()
        diff_y = (grad_pred_y - grad_tgt_y).pow(2).mean()
        return (diff_x + diff_y).item()


metrics = ImageMetrics()

# Evaluate with different quality levels
torch.manual_seed(42)
gt = torch.rand(1, 3, 64, 64)

test_cases = [
    ('Perfect copy', gt.clone()),
    ('Small noise (sigma=0.02)', gt + torch.randn_like(gt) * 0.02),
    ('Medium noise (sigma=0.1)', gt + torch.randn_like(gt) * 0.1),
    ('Large noise (sigma=0.3)', gt + torch.randn_like(gt) * 0.3),
    ('Blurred (5x5 avg)', F.avg_pool2d(F.pad(gt, [2]*4, mode='reflect'), 5, stride=1)),
    ('Random image', torch.rand_like(gt)),
]

print(f"{'Test Case':30s} | {'PSNR (dB)':>10s} | {'SSIM':>8s} | {'LPIPS~':>8s}")
print("-" * 65)

for name, pred in test_cases:
    pred = pred.clamp(0, 1)
    p = metrics.psnr(pred, gt)
    s = metrics.ssim(pred, gt)
    l = metrics.lpips_approx(pred, gt)
    p_str = f'{p:.1f}' if p < 100 else 'inf'
    print(f"{name:30s} | {p_str:>10s} | {s:>8.4f} | {l:>8.4f}")

### 3.2 Benchmark Results (Reference)

Published results on RE10K test set (256x256 resolution):

| Method | PSNR | SSIM | LPIPS | FPS |
|--------|------|------|-------|-----|
| pixelSplat | 25.89 | 0.858 | 0.142 | ~10 |
| MVSplat | 25.97 | 0.869 | 0.128 | ~22 |
| DepthSplat | 26.83 | 0.884 | 0.110 | ~15 |
| **Per-scene 3DGS** | **28+** | **0.92+** | **0.08** | **N/A** |

Note: Per-scene optimization takes minutes per scene, while feed-forward methods run in milliseconds.

In [ ]:
# Visualize benchmark results

methods = ['pixelSplat', 'MVSplat', 'DepthSplat', 'Per-scene\n3DGS']
psnr_vals = [25.89, 25.97, 26.83, 28.5]
ssim_vals = [0.858, 0.869, 0.884, 0.92]
lpips_vals = [0.142, 0.128, 0.110, 0.08]
fps_vals = [10, 22, 15, 0.01]  # Per-scene 3DGS: 0.01 "FPS" (minutes)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
colors = ['#FF9800', '#2196F3', '#9C27B0', '#4CAF50']

# PSNR
ax = axes[0]
bars = ax.bar(methods, psnr_vals, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('PSNR (dB)')
ax.set_title('PSNR (higher = better)', fontsize=11, fontweight='bold')
for bar, val in zip(bars, psnr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}', ha='center', fontsize=9)
ax.set_ylim(24, 30); ax.grid(True, alpha=0.3, axis='y')

# SSIM
ax = axes[1]
bars = ax.bar(methods, ssim_vals, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('SSIM')
ax.set_title('SSIM (higher = better)', fontsize=11, fontweight='bold')
for bar, val in zip(bars, ssim_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', fontsize=9)
ax.set_ylim(0.84, 0.94); ax.grid(True, alpha=0.3, axis='y')

# LPIPS
ax = axes[2]
bars = ax.bar(methods, lpips_vals, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('LPIPS')
ax.set_title('LPIPS (lower = better)', fontsize=11, fontweight='bold')
for bar, val in zip(bars, lpips_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', fontsize=9)
ax.set_ylim(0, 0.2); ax.grid(True, alpha=0.3, axis='y')

# Speed
ax = axes[3]
fps_display = [10, 22, 15, 0.5]
bars = ax.bar(methods, fps_display, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('FPS')
ax.set_title('Speed (higher = better)', fontsize=11, fontweight='bold')
labels = ['10', '22', '15', '~0\n(30min)']
for bar, lbl in zip(bars, labels):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            lbl, ha='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Feed-forward 3DGS Benchmark Comparison (RE10K)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Speed Benchmarking

In [ ]:
# Speed benchmark at different resolutions

def benchmark_model(model, resolutions, num_warmup=3, num_runs=10):
    """Benchmark inference speed at different resolutions."""
    results = []
    model.eval()

    for H, W in resolutions:
        img = torch.randn(1, 3, H, W)
        K = torch.tensor([[50, 0, W/2], [0, 50, H/2], [0, 0, 1]],
                          dtype=torch.float32).unsqueeze(0)

        # Warmup
        with torch.no_grad():
            for _ in range(num_warmup):
                _ = model(img, K)

        # Benchmark
        times = []
        with torch.no_grad():
            for _ in range(num_runs):
                start = time.perf_counter()
                _ = model(img, K)
                end = time.perf_counter()
                times.append((end - start) * 1000)  # ms

        avg_time = np.mean(times)
        std_time = np.std(times)
        fps = 1000.0 / avg_time
        n_gaussians = H * W

        results.append({
            'resolution': f'{H}x{W}',
            'gaussians': n_gaussians,
            'time_ms': avg_time,
            'std_ms': std_time,
            'fps': fps,
        })

    return results


resolutions = [(32, 32), (64, 64), (128, 128), (256, 256)]
bench_results = benchmark_model(model, resolutions)

print(f"{'Resolution':>12s} | {'Gaussians':>10s} | {'Time (ms)':>12s} | {'FPS':>8s}")
print("-" * 55)
for r in bench_results:
    print(f"{r['resolution']:>12s} | {r['gaussians']:>10,} | "
          f"{r['time_ms']:>8.1f} +/- {r['std_ms']:.1f} | {r['fps']:>8.1f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

res_labels = [r['resolution'] for r in bench_results]
times = [r['time_ms'] for r in bench_results]
gaussians = [r['gaussians'] for r in bench_results]

ax = axes[0]
ax.bar(res_labels, times, color='#2196F3', alpha=0.8, edgecolor='black')
ax.set_xlabel('Resolution')
ax.set_ylabel('Inference Time (ms)')
ax.set_title('Inference Time vs Resolution', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.plot(gaussians, times, 'o-', lw=2, markersize=8, color='#2196F3')
ax.set_xlabel('Number of Gaussians (H x W)')
ax.set_ylabel('Inference Time (ms)')
ax.set_title('Scaling: Time vs Gaussians', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Failure Case Analysis

Understanding when feed-forward methods fail is as important as knowing when they succeed.

In [ ]:
# Visualize common failure scenarios

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

failure_cases = [
    ('Textureless Regions',
     'White walls, sky, uniform surfaces',
     'No features to match → ambiguous depth',
     lambda: torch.ones(64, 64, 3) * 0.9 + torch.randn(64, 64, 3) * 0.01),
    ('Repetitive Textures',
     'Tiles, brick walls, checkerboard',
     'Multiple matching candidates → wrong depth',
     lambda: (torch.sin(torch.arange(64).float().unsqueeze(1) * 0.5) *
              torch.cos(torch.arange(64).float().unsqueeze(0) * 0.5)).unsqueeze(-1).expand(-1,-1,3) * 0.3 + 0.5),
    ('Reflections / Transparency',
     'Glass, mirrors, water',
     'Violated multi-view consistency assumption',
     lambda: torch.rand(64, 64, 3) * 0.5 + 0.25),
    ('Large Viewpoint Change',
     'Wide baseline between context views',
     'Large occlusions, few correspondences',
     lambda: torch.zeros(64, 64, 3)),
    ('Dynamic Objects',
     'Moving people, cars, animals',
     'Static scene assumption violated',
     lambda: torch.rand(64, 64, 3)),
    ('Extreme Lighting',
     'Overexposed, underexposed, shadows',
     'Appearance change confuses matching',
     lambda: (torch.rand(64, 64, 3) ** 3)),
]

for i, (title, example, reason, img_fn) in enumerate(failure_cases):
    row, col = i // 3, i % 3
    ax = axes[row, col]
    img = img_fn().clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(title, fontsize=11, fontweight='bold', color='#D32F2F')
    ax.text(0.5, -0.15, f'Example: {example}', transform=ax.transAxes,
            ha='center', fontsize=8, style='italic')
    ax.text(0.5, -0.28, f'Why: {reason}', transform=ax.transAxes,
            ha='center', fontsize=8, color='#666')
    ax.axis('off')

plt.suptitle('Common Failure Cases for Feed-forward 3DGS',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Depth Visualization and Quality Assessment

In [ ]:
# Depth quality analysis

with torch.no_grad():
    result = model(context_images[0:1], K)
    depth_pred = result['depth'][0, 0]  # [H, W]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Depth map
ax = axes[0]
im = ax.imshow(depth_pred.numpy(), cmap='plasma')
ax.set_title('Predicted Depth', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Depth histogram
ax = axes[1]
ax.hist(depth_pred.flatten().numpy(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Depth value')
ax.set_ylabel('Count')
ax.set_title('Depth Distribution', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

# Depth gradient (surface normals approximation)
ax = axes[2]
depth_np = depth_pred.numpy()
grad_x = np.gradient(depth_np, axis=1)
grad_y = np.gradient(depth_np, axis=0)
grad_mag = np.sqrt(grad_x**2 + grad_y**2)
im = ax.imshow(grad_mag, cmap='hot')
ax.set_title('Depth Gradient\n(edge detection)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Opacity distribution
ax = axes[3]
opacities = result['opacities'][0, :, 0].numpy()
ax.hist(opacities, bins=50, color='#FF9800', alpha=0.7, edgecolor='black')
ax.set_xlabel('Opacity')
ax.set_ylabel('Count')
ax.set_title('Opacity Distribution', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Depth and Gaussian Quality Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Depth stats: mean={depth_pred.mean():.2f}, std={depth_pred.std():.2f}, "
      f"range=[{depth_pred.min():.2f}, {depth_pred.max():.2f}]")
print(f"Opacity stats: mean={opacities.mean():.3f}, "
      f"{(opacities > 0.5).sum()}/{len(opacities)} above 0.5")

In [ ]:
# Summary

summary = """
=====================================================================
   Notebook 07 Summary: Inference & Evaluation
=====================================================================

1. INFERENCE PIPELINE
   Input images → Preprocess → Model forward → Gaussians → Render
   Total: ~25ms per novel view for MVSplat (256x256)

2. EVALUATION METRICS
   - PSNR: pixel-level accuracy (dB, higher = better)
   - SSIM: structural similarity ([0,1], higher = better)
   - LPIPS: perceptual quality (lower = better)

3. BENCHMARK RESULTS (RE10K)
   - MVSplat:    25.97 dB / 0.869 SSIM / 22 FPS
   - pixelSplat: 25.89 dB / 0.858 SSIM / 10 FPS
   - DepthSplat: 26.83 dB / 0.884 SSIM / 15 FPS

4. SPEED SCALING
   - Time scales roughly linearly with # Gaussians (H x W)
   - Bottleneck: cost volume construction

5. FAILURE CASES
   - Textureless regions, repetitive textures
   - Reflections, transparency, dynamic objects
   - Wide baselines, extreme lighting

=====================================================================
"""
print(summary)

## What's Next?

**[08_depthsplat_2025_advances.ipynb](./08_depthsplat_2025_advances.ipynb)** - DepthSplat and 2025 advances in feed-forward 3DGS.

---

## References

1. MVSplat: https://arxiv.org/abs/2403.14627
2. pixelSplat: https://arxiv.org/abs/2312.12337
3. LPIPS: https://arxiv.org/abs/1801.03924
4. RE10K: https://google.github.io/realestate10k/